# Pre-commitment swap — residual transfer of the shared code prefix

`p_dst: a'b'd -> d` (patched) · `p_src: a'b'c -> c` (the **residual donor**). Both prefill `a` and `b` and stop at the
frontier's `{var} =`. Everything from the user turn's `<|im_end|>` up to `{var}` is **token-identical**:
```
<|im_end|>\n<|im_start|>assistant\n<code>\ndef func(words: list[str]):\n    a = ...\n    b = ...\n    {var}  |  ' ='  <- pivot (left clean)
```
Over that window, **every** decoder layer's output residual of DST is replaced with SRC's (the pre-commitment-swap
residual transfer, I2, with `p_src` itself as the donor). The DST task list (`d`) is untouched, and so is the eq pivot
` =`: it is not overwritten, but it attends to the swapped k/v from layer 1 on (sanity check below).

* **c** — the patched DST implements SRC's frontier: the pending operation was already committed in the prefix latents
  (pre-commitment effect).
* **d** — the patched DST still implements its own frontier: the pre-commitment is redundant and the goal is re-derived
  from the task span at the pivot (no-op).

Readouts: GoalScope at ` =` on the clean SRC, clean DST, and patched runs (source → destination → patched).

Single source of truth: `src/diff/precommit_swap.py` (`PS.swap_geometry` / `swap_cache` / `swap_patch` / `pivot_sanity` /
`runs`) and the oracle `PP.judge` — exactly what the batch driver `scripts/diff/precommit_swap_eval.py` runs. The notebook
only adds the instance picker, rendering and the knobs. (`RP_INCLUDE_SRC_FRONT` = the script's `--src-front on`.)

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "6")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, collections
from contextlib import nullcontext
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators
import src.diff.path_patch as PP            # PP.judge (the batch eval's oracle)
import src.diff.precommit_swap as PS        # geometry / patch / sanity / runs (single source of truth)
from types import SimpleNamespace

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS = P.GOAL_HEADS
with open(
    os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR), "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")
) as f:
    GOAL_HEADS_250 = [(layer_idx, head_idx) for layer_idx, head_idx in json.load(f)]
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

2026-10-26 12:02:10 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Knobs

In [2]:
#------------------------------------------------------------------------------------------------------
RP_DATATYPE = "list[str]"
RP_SRC_LANG, RP_DST_LANG = "python", "python"
RP_SEED, RP_IDX = 0, 4          # instance: multi_predicate_setups(seed=RP_SEED)[RP_IDX]
RP_N_SAMPLES = 16               # instances sampled (the sweep at the end runs over all of them)
RP_LAYERS = "all"               # "all" or an iterable of layer indices whose OUTPUT residual is transferred
RP_INCLUDE_SRC_FRONT = True    # True: also transfer the frontier description (SRC c-span -> DST d-span), "\n"-padded to equal length
READOUT_HEADS = GOAL_HEADS      # GoalScope heads at ` =` (the path-patching batch eval reads goal_250)
RP_MAX_NEW = 64
RP_ORACLE = True                # PP.judge (causal + readout oracle, claude-sonnet-4-6) — as the batch script
#------------------------------------------------------------------------------------------------------
RP_LAYER_LIST = list(range(mt.n_layer)) if RP_LAYERS == "all" else sorted(RP_LAYERS)
PS_CTX = SimpleNamespace(mt=mt, group=P.GROUP, readout_heads=READOUT_HEADS, gsp=getattr(P, "gsp", None))
VERDICT = {"src": "c · pre-commitment effect", "dst": "d · no-op (pre-commitment redundant)",
           "both": "both (miss)", "neither": "neither"}


def show_windows(tok, title, task_span=None, windows=()):
    # The prompt as the model sees it: task_span -> yellow, transferred windows -> orange, pivot -> salmon outline.
    ids = tok["input_ids"][0].tolist()
    task = set(range(*task_span)) if task_span else set()
    win = {q for lo, hi in windows for q in range(lo, hi)}
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid]).replace("\n", "\\n\n")
        style = ("bold salmon1 underline" if i == len(ids) - 1
                 else "bold black on dark_orange" if i in win else "bright_yellow" if i in task else "")
        t.append(s, style=style)
    rich_print(Panel(t, title=f"[bold]{escape(title)}[/bold]", title_align="left", border_style="grey42",
                     padding=(1, 1)))


def rp_geometry(inst, include_src_front=None):
    # = PS.swap_geometry (+ the instance and token ids, for display). The pivot ` =` is never in the window.
    include_src_front = RP_INCLUDE_SRC_FRONT if include_src_front is None else include_src_front
    g = PS.swap_geometry(inst, include_src_front=include_src_front, ctx=PS_CTX)
    return dict(g, inst=inst, src_ids=PP._ids(g["src_tok"]), dst_ids=PP._ids(g["dst_tok"]))


def rp_cache(g):
    # = PS.swap_cache: SRC's layer outputs over the window — what overwrites DST's.
    return PS.swap_cache(g, RP_LAYER_LIST, ctx=PS_CTX)


def rp_patch(g, cache):
    return PS.swap_patch(g, cache, RP_LAYER_LIST, ctx=PS_CTX)


def rp_run(g, cache, oracle=RP_ORACLE, max_new=RP_MAX_NEW):
    # = PS.runs (clean SRC / clean DST / patched DST generations + GoalScope at ` =`) + PP.judge (the batch oracle).
    out = PS.runs(g, cache, RP_LAYER_LIST, readout_heads=READOUT_HEADS, max_new=max_new, ctx=PS_CTX)
    if oracle:
        causal, readout, _ = PP.judge(g["inst"], out)
        out["ev"] = SimpleNamespace(causal=causal, readout=readout)
    else:
        out["ev"] = None
    return out


## Instance + transferred window

In [3]:
rp_insts = D.multi_predicate_setups(
    RP_DATATYPE, n_conditions=3, n_samples=RP_N_SAMPLES, seed=RP_SEED,
    src_lang=RP_SRC_LANG, dst_lang=RP_DST_LANG, naming="letters", letter_stub="var_eq"
)
for i, s in enumerate(rp_insts):
    print(f"[{i:2d}] shared={[p.id for p in s.shared]}  SRC.frontier={s.expected_next:<22} DST.frontier={s.dst_next}")

g = rp_geometry(rp_insts[RP_IDX])
RP_CFG = (RP_DATATYPE, RP_SRC_LANG, RP_DST_LANG, RP_SEED, RP_IDX, RP_INCLUDE_SRC_FRONT)   # later cells check the knobs still match

_pad = (g["fd0"][1] - g["fd0"][0]) - (g["fs0"][1] - g["fs0"][0])
print(f"\nSRC frontier (c) = {g['c_desc']!r}\nDST frontier (d) = {g['d_desc']!r}")
print(f"frontier spans: SRC {g['fs0'][1] - g['fs0'][0]} tok, DST {g['fd0'][1] - g['fd0'][0]} tok"
      + (f" -> {abs(_pad)} '\\n' pad token(s) before the {'SRC' if _pad > 0 else 'DST'} span" if RP_INCLUDE_SRC_FRONT and _pad
         else "" if RP_INCLUDE_SRC_FRONT else " (not transferred)"))
print(f"transferred {len(g['dst_pos'])} positions × {len(RP_LAYER_LIST)} layers · windows SRC {g['src_win']} -> DST {g['dst_win']} · "
      f"last transferred token {mt.tokenizer.decode([g['dst_ids'][g['dst_pos'][-1]]])!r} · "
      f"pivot {mt.tokenizer.decode([g['dst_ids'][-1]])!r} (pos {g['piv_dst']}, clean)")
show_windows(g["src_tok"], f"SRC · frontier c (yellow) · CACHED-FROM (orange) · L={len(g['src_ids'])}",
             task_span=g["src_fspan"], windows=g["src_win"])
show_windows(g["dst_tok"], f"DST · frontier d (yellow) · PATCHED-TO (orange) · L={len(g['dst_ids'])}",
             task_span=g["dst_fspan"], windows=g["dst_win"])

[ 0] shared=['all_unique_lengths', 'has_palindrome']  SRC.frontier=distinct_count         DST.frontier=count_short
[ 1] shared=['any_empty', 'count_ends_consonant']  SRC.frontier=total_uppercase_letters DST.frontier=distinct_count
[ 2] shared=['total_vowels', 'longest_word_length']  SRC.frontier=count_capitalized      DST.frontier=count_short
[ 3] shared=['longest_word_length', 'count_long']  SRC.frontier=any_empty              DST.frontier=total_length
[ 4] shared=['distinct_count', 'total_uppercase_letters']  SRC.frontier=total_length           DST.frontier=total_vowels
[ 5] shared=['shortest_word_length', 'count_all_lowercase']  SRC.frontier=count_short            DST.frontier=total_length
[ 6] shared=['all_unique_lengths', 'count_starts_vowel']  SRC.frontier=longest_word_length    DST.frontier=shortest_word_length
[ 7] shared=['total_uppercase_letters', 'count_all_lowercase']  SRC.frontier=all_unique_lengths     DST.frontier=count_short
[ 8] shared=['total_length', 'any_empty']  SR

╭─ SRC · frontier c (yellow) · CACHED-FROM (orange) · L=141 ──────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of characters across all words\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · frontier d (yellow) · PATCHED-TO (orange) · L=141 ───────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of distinct words\n                                                                              │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) the total number of vowels across all words\n                                                               │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(set(words))\n                                                                                       │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Sanity check — does the swap reach the (clean) eq pivot ` =`?

The pivot's own residuals are never overwritten, but from layer 1 on its attention reads k/v computed from the swapped
residuals of the window, so its latents should move: **Δ = 0 at layer 0** (layer-0 attention reads the unpatched
embeddings) and **Δ > 0 from layer 1 on**. Two controls:
* every position **before** the first transferred token must be bit-identical to the clean DST run (causal attention);
* the window itself must equal SRC's residuals (by construction).

Last, the next-token prediction at ` =`: clean SRC vs clean DST vs patched.

In [4]:
assert RP_CFG == (RP_DATATYPE, RP_SRC_LANG, RP_DST_LANG, RP_SEED, RP_IDX, RP_INCLUDE_SRC_FRONT), "knobs changed -- re-run the instance cell"
rp_c = rp_cache(g)
san = PS.pivot_sanity(g, rp_c, RP_LAYER_LIST, ctx=PS_CTX)

# control 1: pre-window positions are untouched ; control 2: the window carries SRC's residuals
rich_print(f"control · pre-window positions 0..{min(g['dst_pos']) - 1}: max |h_patched − h_dst| over all layers = "
           f"{san['pre_max_abs']:.3g}  {'[green]OK[/green]' if san['pre_max_abs'] == 0 else '[red]CHANGED[/red]'}")
rich_print(f"control · window: max |h_patched − h_src| = {san['win_max_abs']:.3g}  "
           f"{'[green]OK[/green]' if san['win_max_abs'] < 1e-2 else '[red]MISMATCH[/red]'}")

# the pivot: per-layer ‖h_patched − h_dst‖ / ‖h_dst‖
_rel = san["pivot_rel_delta"]
rich_print(f"pivot ' =' (pos {g['piv_dst']}, not overwritten) · ‖h_patched − h_dst‖₂ / ‖h_dst‖₂ per layer output")
for i in range(0, len(_rel), 8):
    print("  " + "  ".join(f"L{l:02d} {_rel[l]:.3f}" for l in range(i, min(i + 8, len(_rel)))))
rich_print(f"pivot affected: [bold]{san['pivot_affected']}[/bold] · first layer with Δ: {san['first_layer_moved']} · "
           f"layer 0 Δ = {san['pivot_l0_delta']:.3g} · max relative Δ = {san['pivot_max_rel_delta']:.2f} (L{int(np.argmax(_rel)):02d})")

# next-token prediction at the pivot
for tag, k in (("clean SRC (c)", "top_src"), ("clean DST (d)", "top_dst"), ("patched DST", "top_patched")):
    print(f"  next token @ ' =' · {tag:<14} {san[k]}")

control · pre-window positions 0..87: max |h_patched − h_dst| over all layers = 0  OK

control · window: max |h_patched − h_src| = 0  OK

pivot ' =' (pos 140, not overwritten) · ‖h_patched − h_dst‖₂ / ‖h_dst‖₂ per layer output

  L00 0.000  L01 0.003  L02 0.005  L03 0.005  L04 0.007  L05 0.007  L06 0.007  L07 0.007
  L08 0.007  L09 0.008  L10 0.007  L11 0.010  L12 0.010  L13 0.010  L14 0.015  L15 0.018
  L16 0.019  L17 0.019  L18 0.020  L19 0.020  L20 0.022  L21 0.022  L22 0.022  L23 0.023
  L24 0.024  L25 0.029  L26 0.030  L27 0.039  L28 0.041  L29 0.062  L30 0.094  L31 0.097
  L32 0.091  L33 0.088  L34 0.097  L35 0.097  L36 0.104  L37 0.149  L38 0.295  L39 0.341
  L40 0.384  L41 0.430  L42 0.457  L43 0.485  L44 0.509  L45 0.552  L46 0.668  L47 0.712
  L48 0.762  L49 0.822  L50 0.837  L51 0.884  L52 0.896  L53 0.881  L54 0.877  L55 0.847
  L56 0.823  L57 0.798  L58 0.760  L59 0.697  L60 0.639  L61 0.551  L62 0.464  L63 0.617


pivot affected: True · first layer with Δ: 1 · layer 0 Δ = 0 · max relative Δ = 0.90 (L52)

  next token @ ' =' · clean SRC (c)  [(' sum', 0.999), (' len', 0.001), ('(sum', 0.0), ('sum', 0.0), ('\tsum', 0.0)]
  next token @ ' =' · clean DST (d)  [(' sum', 1.0), (' len', 0.0), (" '", 0.0), (' set', 0.0), (' ', 0.0)]
  next token @ ' =' · patched DST    [(' sum', 0.999), (' len', 0.001), ('(sum', 0.0), ('sum', 0.0), ('\tsum', 0.0)]


## Run — generations, GoalScope readouts at ` =`, oracle

In [5]:
assert RP_CFG == (RP_DATATYPE, RP_SRC_LANG, RP_DST_LANG, RP_SEED, RP_IDX, RP_INCLUDE_SRC_FRONT), "knobs changed -- re-run the instance cell"
rp_out = rp_run(g, rp_c)
for tag, col, gen, ro in (("clean SRC (c)", "red1", rp_out["gen_src"], rp_out["ro_src"]),
                          ("clean DST (d)", "sea_green3", rp_out["gen_dst"], rp_out["ro_dst"]),
                          ("patched DST · pre-commitment swap", "salmon1", rp_out["gen_patched"], rp_out["ro_patched"])):
    rich_print(Panel(Text(gen), title=f"[{col}]{tag}[/{col}] · goalscope @ ' =': [italic]{escape(ro)}[/italic]",
                     title_align="left", border_style="grey42"))
print(f"SRC frontier (c) = {g['c_desc']!r}\nDST frontier (d) = {g['d_desc']!r}")
if rp_out["ev"] is not None:
    ev = rp_out["ev"]
    rich_print(f"[bold]causal[/bold]  : patched implements {VERDICT[ev.causal['label']]} — {escape(ev.causal['why'])}")
    r = ev.readout
    rich_print(f"[bold]readout[/bold] : source {r['source']} → destination {r['dest']} → patched {r['patched']} "
               f"[{'valid' if r['valid'] else 'INVALID'} · {'shifted to c' if r['shifted'] else 'not shifted'}]")

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=140 ' =' => I'm counting the total number of characters in the string.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=140 ' =' => I'm counting occurrences of vowels in the string.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

╭─ patched DST · pre-commitment swap · goalscope @ ' =': I'm counting the total number of characters in the strin─╮
│  sum(len(w) for w in words)                                                                                     │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC frontier (c) = 'the total number of characters across all words'
DST frontier (d) = 'the total number of vowels across all words'


causal  : patched implements c · pre-commitment effect — sum(len(w) for w in words) computes the total character 
count across all words, matching sub-goal A (total number of characters), not vowels.

readout : source src → destination dst → patched src

## Sweep — every sampled instance

Same knobs; per instance: the causal verdict (c / d / both / neither) and the readout triple. Readout-shift is over all
instances; *valid* = clean SRC readout says c and clean DST readout says d.

In [6]:
RP_SWEEP_IDX = range(len(rp_insts))   # which instances
rp_rows = []
for i in RP_SWEEP_IDX:
    gi = rp_geometry(rp_insts[i])
    oi = rp_run(gi, rp_cache(gi), oracle=True)
    ev = oi["ev"]
    rp_rows.append(dict(idx=i, c=rp_insts[i].expected_next, d=rp_insts[i].dst_next, causal=ev.causal["label"],
                        ro=(ev.readout["source"], ev.readout["dest"], ev.readout["patched"]),
                        valid=ev.readout["valid"], shifted=ev.readout["shifted"], gen=oi["gen_patched"]))
    rich_print(f"[bold][{i:2d}][/bold] causal={rp_rows[-1]['causal']:<8} readout={'→'.join(rp_rows[-1]['ro'])}")

n = len(rp_rows)
cnt = collections.Counter(r["causal"] for r in rp_rows)
rich_print(f"\n[bold]{RP_DATATYPE} · {RP_SRC_LANG}→{RP_DST_LANG} · n={n} · src_front={RP_INCLUDE_SRC_FRONT}[/bold]")
for lab in ("src", "dst", "both", "neither"):
    print(f"  causal {VERDICT[lab]:<38} {cnt[lab]:3d}  ({cnt[lab] / n:.2f})")
print(f"  readout patched→c (shifted)              {sum(r['shifted'] for r in rp_rows):3d}  ({np.mean([r['shifted'] for r in rp_rows]):.2f})")
print(f"  readout valid (clean SRC→c, DST→d)       {sum(r['valid'] for r in rp_rows):3d}  ({np.mean([r['valid'] for r in rp_rows]):.2f})")
print(f"  patched readout labels: {dict(collections.Counter(r['ro'][2] for r in rp_rows))}")

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=144 ' =' => Identifying unique elements in a list.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=144 ' =' => Counting lines that are shorter than 3 characters.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=144 ' =' => Identifying unique elements in a list.

... (104 outputs left out)

pos=141 ' =' => Counting lines that are shorter than 3 characters.

[15] causal=src      readout=src→dst→src

list · python→python · n=16 · src_front=True

  causal c · pre-commitment effect               16  (1.00)
  causal d · no-op (pre-commitment redundant)     0  (0.00)
  causal both (miss)                              0  (0.00)
  causal neither                                  0  (0.00)
  readout patched→c (shifted)               16  (1.00)
  readout valid (clean SRC→c, DST→d)        16  (1.00)
  patched readout labels: {'src': 16}
